# Estimation de Dimension Fractale par Réseau de Neurones

Ce notebook explore l'estimation de la **dimension intrinsèque** de nuages de points à l'aide d'un réseau de neurones TensorFlow, et compare les résultats avec les méthodes classiques :
- **Two-NN** (Levina-Bickel)
- **Grassberger-Procaccia (GP)**
- **MLE classique**

**Contexte projet** : apprentissage de la dimension fractale à partir de représentations locales de graphes (GNN).

## 1. Imports et configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU disponible : {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Génération de données fractales synthétiques

On génère des variétés de dimensions intrinsèques connues plongées dans un espace ambiant de haute dimension. Chaque variété constitue un exemple d'entraînement.

In [ ]:
def generate_sphere(n_points, intrinsic_dim, ambient_dim=20, noise=0.05):
    """Génère des points sur une hypersphère de dimension intrinsèque d."""
    X = np.random.randn(n_points, intrinsic_dim + 1)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    # Plongement dans l'espace ambiant
    if ambient_dim > intrinsic_dim + 1:
        pad = np.zeros((n_points, ambient_dim - intrinsic_dim - 1))
        X = np.hstack([X, pad])
    # Rotation aléatoire pour rendre le plongement non trivial
    Q, _ = np.linalg.qr(np.random.randn(ambient_dim, ambient_dim))
    X = X @ Q.T
    X += noise * np.random.randn(*X.shape)
    return X

def generate_torus(n_points, ambient_dim=20, noise=0.05):
    """Tore 2D (dimension intrinsèque = 2)."""
    t1 = np.random.uniform(0, 2*np.pi, n_points)
    t2 = np.random.uniform(0, 2*np.pi, n_points)
    R, r = 3.0, 1.0
    x = (R + r * np.cos(t2)) * np.cos(t1)
    y = (R + r * np.cos(t2)) * np.sin(t1)
    z = r * np.sin(t2)
    X = np.column_stack([x, y, z])
    if ambient_dim > 3:
        pad = np.zeros((n_points, ambient_dim - 3))
        X = np.hstack([X, pad])
    Q, _ = np.linalg.qr(np.random.randn(ambient_dim, ambient_dim))
    X = X @ Q.T
    X += noise * np.random.randn(*X.shape)
    return X

def generate_swiss_roll(n_points, ambient_dim=20, noise=0.05):
    """Swiss roll (dimension intrinsèque = 2)."""
    t = 1.5 * np.pi * (1 + 2 * np.random.rand(n_points))
    height = np.random.uniform(0, 10, n_points)
    x = t * np.cos(t)
    y = height
    z = t * np.sin(t)
    X = np.column_stack([x, y, z])
    if ambient_dim > 3:
        pad = np.zeros((n_points, ambient_dim - 3))
        X = np.hstack([X, pad])
    Q, _ = np.linalg.qr(np.random.randn(ambient_dim, ambient_dim))
    X = X @ Q.T
    X += noise * np.random.randn(*X.shape)
    return X

print('Fonctions de génération définies.')

In [ ]:
# Construction du dataset
N_POINTS = 500     # points par nuage
AMBIENT_DIM = 20   # dimension de l'espace ambiant
N_SAMPLES = 200    # nuages par dimension

X_list, y_list = [], []

for d in range(1, 8):  # dimensions intrinsèques 1 à 7
    for _ in range(N_SAMPLES):
        noise_level = np.random.uniform(0.01, 0.1)
        cloud = generate_sphere(N_POINTS, d, AMBIENT_DIM, noise_level)
        X_list.append(cloud)
        y_list.append(d)

# Ajout de tores et swiss rolls (dim=2)
for _ in range(N_SAMPLES // 2):
    X_list.append(generate_torus(N_POINTS, AMBIENT_DIM))
    y_list.append(2)
    X_list.append(generate_swiss_roll(N_POINTS, AMBIENT_DIM))
    y_list.append(2)

print(f'Dataset : {len(X_list)} nuages de points générés')
print(f'Distribution des dimensions : {dict(zip(*np.unique(y_list, return_counts=True)))}')

## 3. Extraction de features locales (Two-NN et distances kNN)

On extrait des **features statistiques** à partir des distances aux k plus proches voisins. Ces features résument la géométrie locale du nuage de points.

In [ ]:
def extract_knn_features(X, k_values=[2, 5, 10, 20]):
    """
    Extrait des features statistiques à partir des distances kNN.
    
    Pour chaque k, calcule :
    - Ratio Two-NN (mu = r2/r1) statistiques
    - Log-distances moyennes
    - Variance des distances
    """
    k_max = max(k_values) + 1
    nbrs = NearestNeighbors(n_neighbors=k_max).fit(X)
    distances, _ = nbrs.kneighbors(X)
    distances = distances[:, 1:]  # Exclure la distance à soi-même
    
    features = []
    
    for k in k_values:
        dists_k = distances[:, :k]
        
        # Log-distances
        log_dists = np.log(dists_k + 1e-10)
        features.extend([
            np.mean(log_dists),
            np.std(log_dists),
            np.median(log_dists),
        ])
        
        # Ratio Two-NN (r2/r1) pour estimation de dim
        if k >= 2:
            mu = dists_k[:, 1] / (dists_k[:, 0] + 1e-10)
            log_mu = np.log(mu + 1e-10)
            features.extend([
                np.mean(log_mu),
                np.std(log_mu),
                np.percentile(log_mu, 25),
                np.percentile(log_mu, 75),
            ])
    
    # Features globales
    r1 = distances[:, 0]
    r2 = distances[:, 1]
    mu_global = r2 / (r1 + 1e-10)
    
    # Estimation Two-NN directe
    mu_filtered = mu_global[mu_global > 1]
    if len(mu_filtered) > 10:
        twonn_estimate = 1.0 / np.mean(np.log(mu_filtered))
    else:
        twonn_estimate = 0.0
    features.append(twonn_estimate)
    
    # Pente log-log (estimation GP simplifiée)
    r_vals = np.percentile(distances[:, 0], [10, 90])
    mask_small = distances[:, 0] < r_vals[1]
    if mask_small.sum() > 5:
        log_r = np.log(distances[mask_small, 0])
        gp_slope = np.polyfit(log_r, np.arange(mask_small.sum()) / mask_small.sum(), 1)[0]
        features.append(np.clip(gp_slope, 0, 20))
    else:
        features.append(0.0)
    
    return np.array(features)


# Test sur un exemple
X_test_cloud = generate_sphere(N_POINTS, 3, AMBIENT_DIM)
feat_test = extract_knn_features(X_test_cloud)
print(f'Dimension du vecteur de features : {len(feat_test)}')

In [ ]:
# Extraction pour tout le dataset
print('Extraction des features (peut prendre ~1 min)...')

features_list = []
for i, cloud in enumerate(X_list):
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(X_list)} nuages traités...')
    features_list.append(extract_knn_features(cloud))

X_features = np.array(features_list)
y_labels = np.array(y_list)

print(f'\nShape features : {X_features.shape}')
print(f'Shape labels   : {y_labels.shape}')

## 4. Méthodes classiques de référence

Implémentation de Two-NN (Levina-Bickel) et Grassberger-Procaccia comme baselines.

In [ ]:
def twonn_estimate(X):
    """Estimateur Two-NN (Levina & Bickel, 2004)."""
    nbrs = NearestNeighbors(n_neighbors=3).fit(X)
    distances, _ = nbrs.kneighbors(X)
    r1 = distances[:, 1]
    r2 = distances[:, 2]
    mu = r2 / (r1 + 1e-12)
    mu = mu[mu > 1]
    if len(mu) < 10:
        return np.nan
    return 1.0 / np.mean(np.log(mu))


def gp_estimate(X, n_pairs=2000, r_range_quantiles=(0.05, 0.5)):
    """
    Estimateur Grassberger-Procaccia (1983).
    Estime la dimension de corrélation via la pente de log C(r) vs log r.
    """
    idx = np.random.choice(len(X), min(500, len(X)), replace=False)
    X_sub = X[idx]
    
    # Calcul des distances par paires
    n = len(X_sub)
    pairs_i = np.random.randint(0, n, n_pairs)
    pairs_j = np.random.randint(0, n, n_pairs)
    mask = pairs_i != pairs_j
    pairs_i, pairs_j = pairs_i[mask], pairs_j[mask]
    
    dists = np.linalg.norm(X_sub[pairs_i] - X_sub[pairs_j], axis=1)
    dists = dists[dists > 0]
    
    r_min = np.quantile(dists, r_range_quantiles[0])
    r_max = np.quantile(dists, r_range_quantiles[1])
    
    r_vals = np.logspace(np.log10(r_min), np.log10(r_max), 20)
    C_vals = np.array([(dists < r).mean() for r in r_vals])
    
    mask = C_vals > 0
    if mask.sum() < 4:
        return np.nan
    
    slope = np.polyfit(np.log(r_vals[mask]), np.log(C_vals[mask]), 1)[0]
    return slope


# Evaluation sur un sous-ensemble
print('Evaluation des méthodes classiques (sous-ensemble)...')
idx_eval = np.random.choice(len(X_list), 50, replace=False)

twonn_preds, gp_preds, true_dims = [], [], []

for i in idx_eval:
    twonn_preds.append(twonn_estimate(X_list[i]))
    gp_preds.append(gp_estimate(X_list[i]))
    true_dims.append(y_labels[i])

twonn_preds = np.array(twonn_preds)
gp_preds = np.array(gp_preds)
true_dims = np.array(true_dims)

mask_valid = ~np.isnan(twonn_preds) & ~np.isnan(gp_preds)
print(f'MAE Two-NN : {np.mean(np.abs(twonn_preds[mask_valid] - true_dims[mask_valid])):.3f}')
print(f'MAE GP     : {np.mean(np.abs(gp_preds[mask_valid] - true_dims[mask_valid])):.3f}')

## 5. Modèle TensorFlow : réseau de neurones pour estimer la dimension

On entraîne un MLP (Multi-Layer Perceptron) sur les features kNN extraites.

In [ ]:
# Normalisation et split train/val/test
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels.astype(np.float32),
    test_size=0.2, random_state=42, stratify=y_labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train : {X_train_s.shape[0]} | Val : {X_val_s.shape[0]} | Test : {X_test_s.shape[0]}')
print(f'Dimension features : {X_train_s.shape[1]}')

In [ ]:
def build_fractal_dim_model(input_dim, dropout_rate=0.3):
    """
    MLP pour la régression de dimension fractale.
    
    Architecture : Features -> Dense -> BN -> Dropout -> ... -> sortie scalaire
    """
    inputs = keras.Input(shape=(input_dim,), name='knn_features')
    
    x = layers.Dense(128, name='dense_1')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    
    x = layers.Dense(256, name='dense_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    
    x = layers.Dense(128, name='dense_3')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout_rate / 2)(x)
    
    x = layers.Dense(64, activation='relu', name='dense_4')(x)
    
    # Connexion résiduelle depuis les features brutes
    shortcut = layers.Dense(64, activation='relu', name='shortcut')(inputs)
    x = layers.Add()([x, shortcut])
    
    # Sortie : dimension positive (softplus pour contraindre > 0)
    outputs = layers.Dense(1, activation='softplus', name='dim_output')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name='FractalDimNet')
    return model


model = build_fractal_dim_model(X_train_s.shape[1])
model.summary()

In [ ]:
# Compilation et callbacks
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='huber',           # Robuste aux outliers vs MSE
    metrics=['mae']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_mae', patience=15, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
    ),
]

# Entraînement
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 6. Visualisation des résultats d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de loss
axes[0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Époque', fontsize=12)
axes[0].set_ylabel('Huber Loss', fontsize=12)
axes[0].set_title('Courbe de perte', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Courbe MAE
axes[1].plot(history.history['mae'], label='Train', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation', linewidth=2)
axes[1].set_xlabel('Époque', fontsize=12)
axes[1].set_ylabel('MAE (dimension)', fontsize=12)
axes[1].set_title('Erreur absolue moyenne', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation et comparaison avec les méthodes classiques

In [ ]:
# Prédictions sur le test set
y_pred_nn = model.predict(X_test_s, verbose=0).flatten()

mae_nn = np.mean(np.abs(y_pred_nn - y_test))
rmse_nn = np.sqrt(np.mean((y_pred_nn - y_test)**2))

print('=== Evaluation sur le test set ===')
print(f'NN  - MAE  : {mae_nn:.4f} | RMSE : {rmse_nn:.4f}')

# Comparaison par dimension vraie
print('\n--- MAE par dimension intrinsèque ---')
for d in sorted(np.unique(y_test)):
    mask = y_test == d
    mae_d = np.mean(np.abs(y_pred_nn[mask] - y_test[mask]))
    print(f'  d={int(d)} : MAE={mae_d:.4f}  (n={mask.sum()})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter : vrai vs prédit
dims_unique = np.unique(y_test)
colors = cm.tab10(np.linspace(0, 1, len(dims_unique)))

for i, d in enumerate(dims_unique):
    mask = y_test == d
    axes[0].scatter(y_test[mask] + np.random.normal(0, 0.05, mask.sum()),
                    y_pred_nn[mask],
                    alpha=0.6, color=colors[i], label=f'd={int(d)}', s=20)

d_range = [0.5, 7.5]
axes[0].plot(d_range, d_range, 'k--', linewidth=2, label='Idéal')
axes[0].set_xlabel('Dimension vraie', fontsize=12)
axes[0].set_ylabel('Dimension prédite (NN)', fontsize=12)
axes[0].set_title('Réseau de neurones : vrai vs prédit', fontsize=13)
axes[0].legend(fontsize=9, ncol=2)
axes[0].grid(True, alpha=0.3)

# Boxplot des erreurs par dimension
errors_by_dim = {}
for d in dims_unique:
    mask = y_test == d
    errors_by_dim[int(d)] = y_pred_nn[mask] - y_test[mask]

bp = axes[1].boxplot(list(errors_by_dim.values()),
                     labels=list(errors_by_dim.keys()),
                     patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Dimension intrinsèque vraie', fontsize=12)
axes[1].set_ylabel('Erreur (prédit - vrai)', fontsize=12)
axes[1].set_title('Distribution des erreurs par dimension', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Extension : architecture GNN pour graphes

Prototype d'un **Graph Neural Network** appliqué directement au graphe kNN du nuage de points. Plus adapté à l'approche du projet avec Vito.

In [ ]:
def build_graph_from_cloud(X, k=10):
    """
    Construit un graphe kNN à partir d'un nuage de points.
    Retourne la matrice d'adjacence et les features de noeuds.
    """
    n = len(X)
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    
    # Features de noeud : coordonnées + statistiques locales
    node_features = np.hstack([
        distances[:, 1:k+1],                          # distances aux k voisins
        np.log(distances[:, 1:k+1] + 1e-10),          # log-distances
        distances[:, 2:k+1] / (distances[:, 1:k] + 1e-10)  # ratios successifs
    ])
    
    # Matrice d'adjacence pondérée (symétrique)
    adj = np.zeros((n, n))
    for i in range(n):
        for j_idx, j in enumerate(indices[i, 1:k+1]):
            w = np.exp(-distances[i, j_idx+1])
            adj[i, j] = w
            adj[j, i] = w
    
    # Normalisation de Laplace
    degree = adj.sum(axis=1, keepdims=True)
    adj_norm = adj / (degree + 1e-10)
    
    return adj_norm, node_features


class GraphConvLayer(layers.Layer):
    """
    Couche de convolution sur graphe (GCN simplifiée).
    H_new = ReLU(A_norm @ H @ W + b)
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dense = layers.Dense(units, use_bias=True)
        self.bn = layers.BatchNormalization()
    
    def call(self, inputs, training=False):
        adj, H = inputs
        # Propagation sur le graphe
        H_agg = tf.matmul(adj, H)
        H_new = self.dense(H_agg)
        H_new = self.bn(H_new, training=training)
        return tf.nn.relu(H_new)


def build_gnn_model(n_nodes, node_feat_dim, n_gcn_layers=3, hidden_dim=64):
    """
    GNN pour l'estimation de dimension fractale.
    
    Pipeline : Node features -> GCN layers -> Global pooling -> MLP -> dim
    """
    adj_input = keras.Input(shape=(n_nodes, n_nodes), name='adjacency')
    feat_input = keras.Input(shape=(n_nodes, node_feat_dim), name='node_features')
    
    H = feat_input
    gcn_layers = []
    
    for i in range(n_gcn_layers):
        gcn = GraphConvLayer(hidden_dim, name=f'gcn_{i+1}')
        gcn_layers.append(gcn)
        H = gcn([adj_input, H])
        H = layers.Dropout(0.2)(H)
    
    # Global pooling : mean + max pooling
    H_mean = layers.GlobalAveragePooling1D()(H)
    H_max  = layers.GlobalMaxPooling1D()(H)
    H_global = layers.Concatenate()([H_mean, H_max])
    
    # MLP final
    x = layers.Dense(64, activation='relu')(H_global)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    output = layers.Dense(1, activation='softplus', name='dim_output')(x)
    
    model = keras.Model(
        inputs=[adj_input, feat_input],
        outputs=output,
        name='FractalGNN'
    )
    return model


# Test de l'architecture
N_NODES_GNN = 100  # subset de noeuds par nuage (pour la mémoire)
K_GNN = 8

X_ex = generate_sphere(N_NODES_GNN, 3, AMBIENT_DIM)
adj_ex, feat_ex = build_graph_from_cloud(X_ex, k=K_GNN)

n_feat_gnn = feat_ex.shape[1]
gnn_model = build_gnn_model(N_NODES_GNN, n_feat_gnn)
gnn_model.summary()

# Test forward pass
adj_batch = adj_ex[np.newaxis, :, :]    # (1, N, N)
feat_batch = feat_ex[np.newaxis, :, :]  # (1, N, F)
pred_test = gnn_model([adj_batch, feat_batch], training=False)
print(f'\nPrédiction test (dim vraie = 3) : {pred_test.numpy()[0,0]:.3f}')

In [ ]:
# Construction du dataset GNN (réduit pour la démonstration)
print('Construction du dataset GNN...')
N_GNN_SAMPLES = 300  # réduit pour rapidité
N_NODES_GNN = 100

adj_list, feat_list_gnn, y_gnn = [], [], []

dims_to_use = [1, 2, 3, 4, 5]
n_per_dim = N_GNN_SAMPLES // len(dims_to_use)

for d in dims_to_use:
    for _ in range(n_per_dim):
        noise = np.random.uniform(0.01, 0.08)
        cloud = generate_sphere(N_NODES_GNN, d, AMBIENT_DIM, noise)
        adj, feat = build_graph_from_cloud(cloud, k=K_GNN)
        adj_list.append(adj)
        feat_list_gnn.append(feat)
        y_gnn.append(d)

adj_array  = np.array(adj_list,  dtype=np.float32)
feat_array = np.array(feat_list_gnn, dtype=np.float32)
y_gnn = np.array(y_gnn, dtype=np.float32)

print(f'adj_array shape  : {adj_array.shape}')
print(f'feat_array shape : {feat_array.shape}')

In [ ]:
# Split et entraînement GNN
idx = np.arange(len(y_gnn))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

gnn_model_train = build_gnn_model(N_NODES_GNN, feat_array.shape[2])
gnn_model_train.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='huber',
    metrics=['mae']
)

history_gnn = gnn_model_train.fit(
    [adj_array[idx_train], feat_array[idx_train]],
    y_gnn[idx_train],
    validation_split=0.15,
    epochs=50,
    batch_size=16,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_mae', patience=10, restore_best_weights=True)
    ],
    verbose=1
)

y_pred_gnn = gnn_model_train.predict(
    [adj_array[idx_test], feat_array[idx_test]], verbose=0
).flatten()

mae_gnn = np.mean(np.abs(y_pred_gnn - y_gnn[idx_test]))
print(f'\nGNN MAE (test) : {mae_gnn:.4f}')

## 9. Tableau comparatif final

In [ ]:
print('='*55)
print(f'{"Méthode":<25} {"MAE":>10} {"Remarque"}')
print('-'*55)
print(f'{"Two-NN":<25} {np.mean(np.abs(twonn_preds[mask_valid] - true_dims[mask_valid])):>10.4f}  Classique, rapide')
print(f'{"Grassberger-Procaccia":<25} {np.mean(np.abs(gp_preds[mask_valid] - true_dims[mask_valid])):>10.4f}  Classique, lent')
print(f'{"MLP (kNN features)":<25} {mae_nn:>10.4f}  Supervisé, features hand-crafted')
print(f'{"GNN (graph-based)":<25} {mae_gnn:>10.4f}  Supervisé, fin-to-end sur graphe')
print('='*55)
print('\nNote : les MAE Two-NN et GP sont sur un sous-ensemble de 50 nuages.')

## 10. Pistes d'extension pour le projet avec Vito

### A explorer :
1. **Features MIC** : ajouter des coefficients de corrélation d'information maximale entre les distances voisines comme features supplémentaires du MLP.
2. **Données réelles** : appliquer sur des représentations de graphes issues de données matériaux (microstructures de composites).
3. **Architecture message-passing** : remplacer la GCN par une architecture MPNN ou GAT (Graph Attention Network) avec `tf.keras` ou `spektral`.
4. **Apprentissage auto-supervisé** : utiliser une loss de contrastive learning pour apprendre des représentations sans labels de dimension.
5. **Incertitude** : ajouter une tête bayésienne (Monte Carlo Dropout) pour quantifier l'incertitude de l'estimation.

### Références clés :
- Levina & Bickel (2004), *Maximum Likelihood Estimation of Intrinsic Dimension*
- Grassberger & Procaccia (1983), *Characterization of Strange Attractors*
- Facco et al. (2017), *Estimating the intrinsic dimension of datasets by a minimal neighborhood information*
- Kipf & Welling (2016), *Semi-Supervised Classification with Graph Convolutional Networks*